In [ ]:
# Discretized (and sequential) HERA NNLSR to investigate bad kernels and boundary effects
# Explore impact of discretized kernels with different reference magnitudes
# Can we make a better discretized kernel only from the data?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy
import scipy.signal as signal
from util.Processing import td_nnlsr_deconvolve, td_convolve, discretize
from util.DataGen import nai_pulse, plastic_pulse, spectrum_trace

from scipy.sparse import lil_matrix, csr_matrix, csc_matrix, coo_matrix, bsr_matrix
from scipy.optimize import nnls, least_squares
from scipy.linalg import circulant

from scipy.ndimage import gaussian_filter1d

In [ ]:
def nai_pulse_mod(a, n=-1, sampling_ratio=1, b=.23e7, c=.145e7, d=.053e7):
    sos1 = signal.butter(3, b, btype='low', analog=False, output='sos', fs=40e6)
    sos2 = signal.butter(2, c, btype='low', analog=False, output='sos', fs=40e6)
    sos3 = signal.butter(1, d, btype='low', analog=False, output='sos', fs=40e6)
 
    dt = 25e-9 /sampling_ratio #TODO implement sampling ratio...
    start = 0
    if n == -1:
        stop = 2.5e-6
    else:
        stop = n * dt
    t = np.arange(start, stop, dt)
    # t = np.arange(0., 2.5e-6, 25e-9)
    y = t * 0
    y[4] = 1
    fil1 = signal.sosfilt(sos1, y)
    fil1 = fil1*a/np.max(fil1)
    fil2 = signal.sosfilt(sos2, y)
    fil2 = fil2*a/np.max(fil2)
    fil3 = signal.sosfilt(sos3, y)
    fil3 = fil3*a/np.max(fil3)
    fil = np.concatenate((fil1[0:11], fil2[11:17], fil3[17:]))
    return t, fil

In [ ]:
# Trace
n_photons = 50
trace_len = 1000
kernel_ref_magnitude = 1000
bits = 8 # 8 bits is

sensor_type = 'NaI'
seed = 3
noise_stdev = 0 # mV

dt = np.float64(25e-9) #sampling rate in seconds. 40MHz
total_time = dt * trace_len
count_rate = n_photons / total_time
print('Time, rate = {}, {}'.format(total_time, count_rate))

_, decimal_kernel = nai_pulse(1, 101, 1)
spectrum = np.loadtxt('../original/NaI_Response',usecols=(1), dtype=float)
bins = 1E3 * np.loadtxt('../original/NaI_Response',usecols=(0), dtype=float) # in kev

plt.figure(figsize=(3, 3), dpi=200)
plt.plot(decimal_kernel, label='True Kernel')
discretized_kernel = discretize(nai_pulse(kernel_ref_magnitude, 101)[1], bits)/kernel_ref_magnitude
plt.plot(discretized_kernel, label='Discretized Kernel')
plt.legend()


keV_per_area = .147 #determined by trial and error to match energy range of instrument 
mV_per_ADC = 1000./4096.
area_per_peak = np.sum(discretized_kernel)/max(discretized_kernel)
mV_per_keV = mV_per_ADC/(keV_per_area*area_per_peak)
print('mV_per_keV', mV_per_keV)

trace, time_vector, volts_list, volts_time_index = spectrum_trace(
                                                    count_rate_=count_rate, dt_=dt, total_time_=total_time,
                                                    pulse_=discretized_kernel, bin_energies_=bins, spectrum_=spectrum,
                                                    mV_per_keV_=mV_per_keV, noise_std_=noise_stdev, baseline_=0,
                                                    sampling_ratio_=1, discretize=True, bits_=bits,
                                                    seed_=seed, clip=False, debug=False)
print(trace.shape, time_vector.shape, volts_list.shape, volts_time_index.shape)


In [ ]:
energy_vector = np.zeros_like(trace)
energy_vector[volts_time_index] = volts_list


nnlsr_size = min(1000, trace.size)
blocks = (trace.size + nnlsr_size - 1) // nnlsr_size

deconvolved = []
for i in range(blocks):
    data = trace[i * nnlsr_size : (i + 1) * nnlsr_size]
    deconvolved.append(td_nnlsr_deconvolve(data, discretized_kernel))
deconvolved = np.concatenate(deconvolved, axis=0)

print(energy_vector.shape, energy_vector.shape, trace.shape)

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(10, 10), dpi=200)
fig.suptitle('Deconvolution error using Dense NNLSR')

block_markers = np.arange(blocks + 1) * nnlsr_size

hist, _ = np.histogram(energy_vector[energy_vector>0], bins)
centers = bins[1:] + np.diff(bins)
axes[0].plot(centers, hist, 'g.', markersize=20,
             label='True Distribution')
hist, _ = np.histogram(deconvolved[deconvolved>0], bins=bins)
centers = bins[1:] + np.diff(bins)
axes[0].plot(centers, hist, 'r.', label='Deconvolved Distribution')
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].legend()

error = np.abs(energy_vector - deconvolved)
axes[1].plot(error, 'r.', label='Abs Deconvolution Error')
axes[1].set_yscale('log')
axes[1].plot(block_markers, 1E-4  * np.ones_like(block_markers), '|', label='NNLSR Block Edges')
axes[1].set_ylim(1E-4, 1E3)
axes[1].set_ylabel('Volts')
axes[1].legend()
axes[1].set_xlim([0, trace.size])

axes[2].plot(trace, label='Trace')
mask = energy_vector > 0
axes[2].plot(np.arange(energy_vector.size)[mask], energy_vector[mask], color='red', marker='*', linestyle='', label='Events')
axes[2].plot(block_markers, -5 * np.ones_like(block_markers), 'r|', label='NNLSR Block Edges')
axes[2].legend(loc=2)
axes[2].set_xlim([0, trace.size])
axes[2].set_ylim(-5, 100)
axes[2].set_ylabel('Volts')

axes[3].plot(trace, label='Trace')
mask = deconvolved > 0
axes[3].plot(np.arange(deconvolved.size)[mask], deconvolved[mask], color='red', marker='*', linestyle='', label='Deconvolved Events')
axes[3].plot(block_markers, -5 * np.ones_like(block_markers), 'r|', label='NNLSR Block Edges')
axes[3].legend(loc=2)
axes[3].set_xlim([0, trace.size])
axes[3].set_ylim(-5, 100)
axes[3].set_ylabel('Volts')

error_gt1 = error > 1
error_lt1 = error < 1
axes[4].plot(np.arange(trace.size)[error_gt1], deconvolved[error_gt1],
             color='red', marker='.', linestyle='', label='Deconvolved Error > 1', zorder=2)
axes[4].plot(np.arange(trace.size)[error_gt1], energy_vector[error_gt1],
             color='green', marker='.', markersize=10, linestyle='',
             label='Error > 1 Truth', zorder=1)

axes[4].plot(np.arange(trace.size)[error_lt1], deconvolved[error_lt1],
             color='blue', marker='.', linestyle='', label='Deconvolved Error < 1',
             alpha=1, zorder=0)
axes[4].plot(block_markers, -5 * np.ones_like(block_markers), 'r|', label='NNLSR Block Edges')
axes[4].set_ylabel('Volts')
axes[4].set_xlabel('Time Index')
axes[4].legend(loc=2)
axes[4].set_xlim([0, trace.size])

plt.show()

In [ ]:
def nnlsr_sparse_weights_with_pad(kernel_, time_len):
    # returns sparse weight matrix of extended diagonal of weights.
    #         Weights should be used with a trace padded with zeros of
    #         at least length of kernel.
    #
    # param: kernel_ the unscaled kernel shape
    # param: time_len: The length of time to solve weights for.
    # returns: scipy.sparse.csr_matrix

    assert time_len > kernel_.size

    k = kernel_
    nk = kernel_.size
    t = time_len
    n_padded = t + nk

    # Start with COO matrix (define with coordinates)
    data = np.tile(k, n_padded).reshape(k.size, n_padded)
    i = np.tile(np.arange(n_padded), nk).reshape((nk, n_padded)) + np.arange(nk)[:, None]
    j = np.tile(np.arange(n_padded), nk).reshape(nk, n_padded)
    matrix = coo_matrix((data.ravel(), (j.T.ravel(),i.T.ravel())), (n_padded + nk + 1, n_padded + nk + 1))
    matrix = lil_matrix(matrix) # make subscriptable to slice to desired size and values

    # bsr matrix efficient for math operations, reccomended by scipy for new general work
    final_matrix = bsr_matrix(matrix[:n_padded, :n_padded])
    del matrix, data, i ,j
    return final_matrix




In [ ]:
# good

# here we prevented photons from being chosen using an internal pad
C = nnlsr_sparse_weights_with_pad(discretized_kernel, trace.size - discretized_kernel.size).T
n_trace = C.shape[0]
C = csc_matrix(lil_matrix(C))
sparsity = C


def residual(x, C, trace):
    deconv_estimate = C @ x   
    # deconv_estimate[deconv_estimate>100]=100
    # deconv_estimate = discretize(deconv_estimate, bits=bits) # this does not work at all...
    out = trace - deconv_estimate
    
    # assert False
    return out

# TODO these are not quite correct... needs to be loss = (residual)**2 + L1(weights) not L1(residual)
# looking in trf.py, it seems we actually cant do L1 regularization, because the internal functions seperate out
# residual and loss calcualtions, where we essential need y an X in the same function together. Could copy here and
# modify... (commit on new branch before modifying) or could try to use scipy.optimize.minimize like here:
# https://stats.stackexchange.com/questions/573631/linear-regression-with-lasso-regularization-by-using-scikitlearn-and-scipy-optim
# this might be simpler anyway, with more options
# note in the above solution, they got around the problem by supplying y as a global, alongside the normal function arg X (Z, there).
# note scipy.optimize.minimize has perfromed worse...
def pseudo_l1(z):
    # loss = z
    # d_loss = np.ones_like(z)
    # dd_loss = 2 * np.zeros_like(z)
    # return np.stack((loss, d_loss, dd_loss))
    
    loss = z**2 + z
    # print("Loss:", np.sum(loss))
    d_loss = 2 * z + np.ones_like(z)
    dd_loss = 2 * np.ones_like(z)
    return np.stack((loss, d_loss, dd_loss)) 

#TODO maybe the sparse matrix type of C changes the speed this runs...
#   seems like ListOfLists could be faster by directly providing inedeces...

clip_value = 100 # probably 1000 with real sensor, just nice to play with value here as demo
padded_trace = trace.copy()
# padded_trace[padded_trace > clip_value] = clip_value
# padded_trace = gaussian_filter1d(padded_trace, sigma=.5) # not helpful, just spreads the estimation. Anti-L1
x0 = np.ones_like(trace)

solution1 = least_squares(fun=residual,
                         jac='cs',
                         method='trf', # use with bounds!
                         bounds=[0, np.inf],
                         args=(C, padded_trace),
                         x0=x0,
                         x_scale=10, # it seems that a larger diff step + x_scale='jac' together can be an issue
                         diff_step=.1,
                         jac_sparsity=sparsity,   # not necessary, but speeds things up a lot which will be useful for scaling to high n
                         ftol=1E-10,
                         # gtol=1E5,
                         loss=pseudo_l1, # if not callable, 'linear', 'soft_l1' or 'huber'
                         max_nfev=200,
                         verbose=2
                         )


In [ ]:
energy_vector = np.zeros_like(padded_trace)
energy_vector[volts_time_index] = volts_list

fig, axes = plt.subplots(5, 1, figsize=(10, 10), dpi=200)
fig.suptitle('Deconvolution error using Sparse NNLSR')

block_markers = np.arange(blocks + 1) * nnlsr_size

hist, _ = np.histogram(energy_vector[energy_vector>0], bins)
centers = bins[1:] + np.diff(bins)
axes[0].plot(centers, hist, 'g.', markersize=20, label='True Distribution')
hist, _ = np.histogram(solution1.x[solution1.x>0], bins=bins)
centers = bins[1:] + np.diff(bins)
axes[0].plot(centers, hist, 'r.', label='Deconvolved Distribution')
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].legend()

error = np.abs(energy_vector - solution1.x)
axes[1].plot(error, 'r.', label='Abs Deconvolution Error')
axes[1].set_yscale('log')
axes[1].plot(block_markers, 1E-4  * np.ones_like(block_markers), '|', label='NNLSR Block Edges')
axes[1].set_ylim(1E-4, 1E3)
axes[1].set_ylabel('Volts')
axes[1].set_xlim([0, padded_trace.size])
axes[1].legend()

axes[2].plot(padded_trace, label='Trace')
mask = energy_vector > 0
axes[2].plot(np.arange(energy_vector.size)[mask], energy_vector[mask], color='red', marker='*', linestyle='', label='Events')
axes[2].plot(block_markers, -5 * np.ones_like(block_markers), 'r|', label='NNLSR Block Edges')
axes[2].legend(loc=2)
axes[2].set_xlim([0, padded_trace.size])
axes[2].set_ylim(-5, 110)
axes[2].set_ylabel('Volts')

axes[3].plot(padded_trace, label='Trace')
mask = solution1.x > 1
axes[3].plot(np.arange(solution1.x.size)[mask], solution1.x[mask], color='red', marker='*', linestyle='', label='Deconvolved Events')
axes[3].plot(block_markers, -5 * np.ones_like(block_markers), 'r|', label='NNLSR Block Edges')
axes[3].legend(loc=2)
axes[3].set_xlim([0, padded_trace.size])
axes[3].set_ylim(-5, 110)
axes[3].set_ylabel('Volts')


error_gt1 = error > 1
error_lt1 = error < 1
axes[4].plot(np.arange(padded_trace.size)[error_gt1], solution1.x[error_gt1],
             color='red', marker='.', linestyle='', label='Deconvolved: Error > 1', zorder=2)
axes[4].plot(np.arange(padded_trace.size)[error_gt1], energy_vector[error_gt1],
             color='green', marker='.', markersize=10, linestyle='',
             label='Error > 1 Truth', zorder=1)

axes[4].plot(np.arange(padded_trace.size)[error_lt1], solution1.x[error_lt1],
             color='blue', marker='.', linestyle='', label='Deconvolved: Error < 1',
             alpha=1, zorder=0)
axes[4].plot(block_markers, -5 * np.ones_like(block_markers), 'r|', label='NNLSR Block Edges')
axes[4].set_ylabel('Volts')
axes[4].set_xlabel('Time Index')
axes[4].legend(loc=2)
axes[4].set_xlim([0, padded_trace.size])

plt.show()

In [ ]:
# Lasso does not work but other sklearn linear models do... why?
# is this a fundamental math issue or an optimization issue?

from sklearn.linear_model import Lasso, LinearRegression, Ridge

X = nnlsr_sparse_weights_with_pad(discretized_kernel, trace.size - discretized_kernel.size).T
y = trace

lasso = Lasso(alpha=1E-2,
              fit_intercept=False,
              precompute=False, #precomupte gram matrix. Maybe this is faster?
              copy_X=True,
              max_iter=int(1E7),
              tol=1E-15,
              warm_start=False, # When set to True, reuse the solution of the previous call to fit as initialization,
              positive=True,
              random_state=0,
              selection='cyclic') # random much faster for high tolerance (random, cyclic)
lasso.fit(X=X,y=y)
deconv = lasso.coef_
print(deconv.shape)

# reg = LinearRegression(fit_intercept=False,
#                        copy_X=True, 
#                        n_jobs=None,
#                        positive=True)
# reg.fit(X=X.toarray(), y=y)
# deconv = reg.coef_
# print(deconv.shape)

# reg = Ridge(alpha=1E-5,
#             fit_intercept=False,
#             copy_X=True,
#             max_iter=None,
#             tol=1E-10,
#             solver='auto',
#             positive=True,
#             random_state=None,
#             )
# 
# reg.fit(X=X.toarray(), y=y)
# deconv = reg.coef_
# print(deconv.shape)


fig, axes = plt.subplots(5, 1, figsize=(10, 10), dpi=200)
fig.suptitle('Deconvolution error using Lasso')

hist, _ = np.histogram(energy_vector[energy_vector>0], bins)
centers = bins[1:] + np.diff(bins)

error = np.abs(energy_vector - deconv)
axes[0].plot(error, 'r.', label='Abs Deconvolution Error')
axes[0].set_yscale('log')
# axes[0].plot(block_markers, 1E-4  * np.ones_like(block_markers), '|', label='NNLSR Block Edges')
axes[0].set_ylim(1E-4, 1E3)
axes[0].set_ylabel('Volts')
axes[0].set_xlim([0, padded_trace.size])
axes[0].legend()

# Error by difference between photons
right=axes[0].twinx()
right.plot(volts_time_index[:-1], np.diff(volts_time_index), color='green', markersize=2, marker='.', linestyle='', label='diff(Photon Times)')
gt0 = np.where(deconv > 0)[0]
diff_gt0 = np.diff(gt0)
right.plot(np.arange(padded_trace.size)[gt0][:-1], diff_gt0, color='blue', markersize=2, marker='.', linestyle='', label='diff(Where(deconv>0))')
right.set_ylim(-4, 8)
right.set_ylabel('Offset')
right.legend()

axes[1].plot(centers, hist, 'g.', markersize=20, label='True Distribution')
hist, _ = np.histogram(deconv[deconv>0], bins=bins)
centers = bins[1:] + np.diff(bins)
axes[1].plot(centers, hist, 'r.', label='Deconvolved Distribution')
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].legend()



axes[2].plot(padded_trace, label='Trace')
mask = energy_vector > 0
axes[2].plot(np.arange(energy_vector.size)[mask], energy_vector[mask], color='red', marker='*', linestyle='', label='Events')
axes[2].plot(block_markers, -5 * np.ones_like(block_markers), 'r|', label='NNLSR Block Edges')
axes[2].legend(loc=2)
axes[2].set_xlim([0, padded_trace.size])
axes[2].set_ylim(-5, 110)
axes[2].set_ylabel('Volts')

axes[3].plot(padded_trace, label='Trace')
mask = deconv > 1
axes[3].plot(np.arange(deconv.size)[mask], deconv[mask], color='red', marker='*', linestyle='', label='Deconvolved Events')
axes[3].plot(block_markers, -5 * np.ones_like(block_markers), 'r|', label='NNLSR Block Edges')
axes[3].legend(loc=2)
axes[3].set_xlim([0, padded_trace.size])
axes[3].set_ylim(-5, 110)
axes[3].set_ylabel('Volts')


error_gt1 = error > 1
error_lt1 = error < 1
axes[4].plot(np.arange(padded_trace.size)[error_gt1], deconv[error_gt1],
             color='red', marker='.', linestyle='', label='Deconvolved: Error > 1', zorder=2)
axes[4].plot(np.arange(padded_trace.size)[error_gt1], energy_vector[error_gt1],
             color='green', marker='.', markersize=10, linestyle='',
             label='Error > 1 Truth', zorder=1)

axes[4].plot(np.arange(padded_trace.size)[error_lt1], deconv[error_lt1],
             color='blue', marker='.', linestyle='', label='Deconvolved: Error < 1',
             alpha=1, zorder=0)
axes[4].plot(block_markers, -5 * np.ones_like(block_markers), 'r|', label='NNLSR Block Edges')
axes[4].set_ylabel('Volts')
axes[4].set_xlabel('Time Index')
axes[4].legend(loc=2)
axes[4].set_xlim([0, padded_trace.size])


In [ ]:
# Lasso does not work but other sklearn linear models do... why?
# is this a fundamental math issue or an optimization issue?

from sklearn.linear_model import Lasso, LinearRegression, Ridge

X = nnlsr_sparse_weights_with_pad(discretized_kernel, trace.size - discretized_kernel.size).T
y = trace

lasso = Lasso(alpha=1E-2,
              fit_intercept=False,
              precompute=False, #precomupte gram matrix. Maybe this is faster?
              copy_X=True,
              max_iter=int(1E7),
              tol=1E-15,
              warm_start=False, # When set to True, reuse the solution of the previous call to fit as initialization,
              positive=True,
              random_state=0,
              selection='cyclic') # random much faster for high tolerance (random, cyclic)
lasso.fit(X=X,y=y)
deconv = lasso.coef_
print(deconv.shape)

fig, axes = plt.subplots(6, 1, figsize=(10, 10), dpi=200)
fig.suptitle('Deconvolution error using Lasso')

hist, _ = np.histogram(energy_vector[energy_vector>0], bins)
centers = bins[1:] + np.diff(bins)

error = np.abs(energy_vector - deconv)
axes[0].plot(error, 'r.', label='Abs Deconvolution Error')
axes[0].set_yscale('log')
# axes[0].plot(block_markers, 1E-4  * np.ones_like(block_markers), '|', label='NNLSR Block Edges')
axes[0].set_ylim(1E-4, 1E3)
axes[0].set_ylabel('Volts')
axes[0].set_xlim([0, padded_trace.size])
axes[0].legend()

# Error by difference between photons
right=axes[0].twinx()
right.plot(volts_time_index[:-1], np.diff(volts_time_index), color='green', markersize=2, marker='.', linestyle='', label='diff(Photon Times)')
gt0 = np.where(deconv > 0)[0]
diff_gt0 = np.diff(gt0)
right.plot(np.arange(padded_trace.size)[gt0][:-1], diff_gt0, color='blue', markersize=2, marker='.', linestyle='', label='diff(Where(deconv>0))')
right.set_ylim(-4, 8)
right.legend()


axes[1].plot(centers, hist, 'g.', markersize=20, label='True Distribution')
hist, _ = np.histogram(deconv[deconv>0], bins=bins)
centers = bins[1:] + np.diff(bins)
axes[1].plot(centers, hist, 'r.', label='Deconvolved Distribution')
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].legend()

axes[2].plot(padded_trace, label='Trace')
mask = energy_vector > 0
axes[2].plot(np.arange(energy_vector.size)[mask], energy_vector[mask], color='red', marker='*', linestyle='', label='Events')
axes[2].plot(block_markers, -5 * np.ones_like(block_markers), 'r|')
axes[2].legend(loc=2)
axes[2].set_xlim([0, padded_trace.size])
axes[2].set_ylim(-5, 110)
axes[2].set_ylabel('Volts')

axes[3].plot(padded_trace, label='Trace')
mask = deconv > 1
axes[3].plot(np.arange(deconv.size)[mask], deconv[mask], color='red', marker='*', linestyle='', label='Deconvolved Events')
axes[3].plot(block_markers, -5 * np.ones_like(block_markers), 'r|')
axes[3].legend(loc=2)
axes[3].set_xlim([0, padded_trace.size])
axes[3].set_ylim(-5, 110)
axes[3].set_ylabel('Volts')

# sum consecutive numbers left. Can we do this is one line or do we need a look for > 2 in a  row?# print(gt0)
def gather_consecutive(vector, print_=False):
    # Gather consecutive indeces. Intended for a situation whre non-zero values are sparse.
    # this will be nonsense with many non-zeros
    consecutive_groups = []
    non_consecutive = []
    
    diff = np.diff(vector)
    i = 0
    while i < diff.size:
        consecutive = 0
        while i+consecutive < diff.size and diff[i+consecutive] == 1:
            consecutive += 1
            
        if consecutive == 0:
            i += 1
            non_consecutive.append(vector[i])
            continue
        else:
            group = vector[i:i+consecutive+1]
            consecutive_groups.append(group)
            if print_:
                print(group)
            if i+consecutive+1 >= diff.size:
                break
            i+=consecutive
    return consecutive_groups, non_consecutive 


error_gt1 = error > 1
error_lt1 = error < 1
axes[5].plot(np.arange(padded_trace.size)[error_gt1], deconv[error_gt1],
             color='red', marker='.', linestyle='', label='Deconvolved: Error > 1', zorder=2)
axes[5].plot(np.arange(padded_trace.size)[error_gt1], energy_vector[error_gt1],
             color='green', marker='.', markersize=10, linestyle='',
             label='Error > 1 Truth', zorder=1)

axes[5].plot(np.arange(padded_trace.size)[error_lt1], deconv[error_lt1],
             color='blue', marker='.', linestyle='', label='Deconvolved: Error < 1',
             alpha=1, zorder=0)
axes[5].plot(block_markers, -5 * np.ones_like(block_markers), 'r|', label='NNLSR Block Edges')
axes[5].set_ylabel('Volts')
axes[5].set_xlabel('Time Index')
axes[5].legend(loc=2)
axes[5].set_xlim([0, padded_trace.size])

plt.figure(figsize=(3,1), dpi=200)
plt.hist(np.diff(np.where(deconv>0)[0]), bins=np.arange(0, 10))

In [ ]:
print(volts_list.size, volts_time_index.size)

consecutive, nonconsecutive = gather_consecutive(gt0, print_=False)

new_indeces = []
new_volts = []
for group in consecutive:
    new_indeces.append(group[0])
    new_volts.append(np.sum(deconv[group]))
    
print(new_indeces)
new_indeces.extend(nonconsecutive)
print(new_indeces)
print(len(new_indeces), np.unique(new_indeces).size)

new_volts.extend(deconv[nonconsecutive])



new_indeces = np.array(new_indeces)
new_volts = np.array(new_volts)

sorting_index = np.argsort(new_indeces)
new_indeces = new_indeces[sorting_index]
new_volts = new_volts[sorting_index]
# print(new_indeces)


fig, axes = plt.subplots(2, 1, figsize=(10, 4), dpi=200)
axes[0].plot(padded_trace, label='Trace')
mask = deconv > 1
axes[0].plot(np.arange(energy_vector.size)[mask], energy_vector[mask], color='green', marker='*', linestyle='', label='Events')
axes[0].plot(np.arange(deconv.size)[mask], deconv[mask], color='red', marker='.', linestyle='', label='Deconvolved Events')

axes[0].legend(loc=2)
axes[0].set_xlim([0, padded_trace.size])
axes[0].set_ylim(-5, 110)
axes[0].set_ylabel('Volts')

axes[1].plot(padded_trace, label='Trace')
axes[1].plot(np.arange(energy_vector.size)[mask], energy_vector[mask], color='green', marker='*', linestyle='', label='Events')
axes[1].plot(new_indeces, new_volts, color='red', marker='.', linestyle='', label='Cons. Summed Deconv.')
axes[1].legend(loc=2)
axes[1].set_xlim([0, padded_trace.size])
axes[1].set_ylim(-5, 110)
axes[1].set_ylabel('Volts')
        

In [ ]:
# # scipy.optimize, minimize does not work well,perhaps this problem is too non-convex?
# # tried all the parameters relevant to bounded problem... especially  metod and jac.
# 
# from scipy.optimize import minimize
# 
# # here we prevented photons from being chosen using an internal pad
# C = nnlsr_sparse_weights_with_pad(discretized_kernel, trace.size - discretized_kernel.size).T
# n_trace = C.shape[0]
# C = csc_matrix(lil_matrix(C))
# sparsity = C
# 
# padded_trace = trace
# x0 = np.ones_like(trace)
# 
# alpha = 0
# def l1(xhat, C, trace):
#     # print(xhat.dtype, trace.dtype)
#     error = trace - C @ xhat
# 
#     sqe = np.sum(error**2)
#     # sqe = np.sum(np.abs(error))
#     # sqe = -np.sum(error)
#     sum_l1 = alpha * np.sum(xhat)
# 
#     #TODO note these are complex numbers...? why?
# 
#     print('Error: {}, Loss: {}'.format(sqe, sum_l1))
# 
#     # assert False
#     return sqe + sum_l1
# 
# solution1 = minimize(fun=l1,
#                          jac='3-point', # cs seems to force complex numbers. Docs say that ojective needs to handle complex num correcrtly for this...
#                          # method='Nelder-Mead', # truly glacial
#                          method='L-BFGS-B', # fast, does nto seem to converge onto correct soln. Error = 1783 at end...                      
#                          # method='Powell', # pretty slow, but not as bad as Nelder... Seems to frequnelty but temporarily
#                                           # get caught in local minima or low grad regions. Seemed to crash silently at the end...
#                          # method = 'TNC', # slow, bad
#                          bounds=[(0, np.inf)] * x0.size,
#                          args=(C, trace),
#                          x0=x0,
#                          tol=1E-20,
#                          callback=None, # might be nice for some intermediate outputs
#                          )
# energy_vector = np.zeros_like(padded_trace)
# energy_vector[volts_time_index] = volts_list
# 
# fig, axes = plt.subplots(5, 1, figsize=(10, 10), dpi=200)
# fig.suptitle('Deconvolution error using Sparse NNLSR')
# 
# block_markers = np.arange(blocks + 1) * nnlsr_size
# 
# hist, _ = np.histogram(energy_vector[energy_vector>0], bins)
# centers = bins[1:] + np.diff(bins)
# axes[0].plot(centers, hist, 'g.', markersize=20, label='True Distribution')
# hist, _ = np.histogram(solution1.x[solution1.x>0], bins=bins)
# centers = bins[1:] + np.diff(bins)
# axes[0].plot(centers, hist, 'r.', label='Deconvolved Distribution')
# axes[0].set_xscale('log')
# axes[0].set_yscale('log')
# axes[0].legend()
# 
# 
# error = np.abs(energy_vector - solution1.x)
# axes[1].plot(error, 'r.', label='Abs Deconvolution Error')
# axes[1].set_yscale('log')
# axes[1].plot(block_markers, 1E-4  * np.ones_like(block_markers), '|', label='NNLSR Block Edges')
# axes[1].set_ylim(1E-4, 1E3)
# axes[1].set_ylabel('Volts')
# axes[1].set_xlim([0, padded_trace.size])
# axes[1].legend()
# 
# axes[2].plot(padded_trace, label='Trace')
# mask = energy_vector > 0
# axes[2].plot(np.arange(energy_vector.size)[mask], energy_vector[mask], color='red', marker='*', linestyle='', label='Events')
# axes[2].plot(block_markers, -5 * np.ones_like(block_markers), 'r|', label='NNLSR Block Edges')
# axes[2].legend(loc=2)
# axes[2].set_xlim([0, padded_trace.size])
# axes[2].set_ylim(-5, 100)
# axes[2].set_ylabel('Volts')
# 
# axes[3].plot(padded_trace, label='Trace')
# mask = solution1.x > 1
# axes[3].plot(np.arange(solution1.x.size)[mask], solution1.x[mask], color='red', marker='*', linestyle='', label='Deconvolved Events')
# axes[3].plot(block_markers, -5 * np.ones_like(block_markers), 'r|', label='NNLSR Block Edges')
# axes[3].legend(loc=2)
# axes[3].set_xlim([0, padded_trace.size])
# axes[3].set_ylim(-5, 100)
# axes[3].set_ylabel('Volts')
# 
# 
# error_gt1 = error > 1
# error_lt1 = error < 1
# axes[4].plot(np.arange(padded_trace.size)[error_gt1], solution1.x[error_gt1],
#              color='red', marker='.', linestyle='', label='Deconvolved: Error > 1', zorder=2)
# axes[4].plot(np.arange(padded_trace.size)[error_gt1], energy_vector[error_gt1],
#              color='green', marker='.', markersize=10, linestyle='',
#              label='Error > 1 Truth', zorder=1)
# 
# axes[4].plot(np.arange(padded_trace.size)[error_lt1], solution1.x[error_lt1],
#              color='blue', marker='.', linestyle='', label='Deconvolved: Error < 1',
#              alpha=1, zorder=0)
# axes[4].plot(block_markers, -5 * np.ones_like(block_markers), 'r|', label='NNLSR Block Edges')
# axes[4].set_ylabel('Volts')
# axes[4].set_xlabel('Time Index')
# axes[4].legend(loc=2)
# axes[4].set_xlim([0, padded_trace.size])
# 
# plt.show()


In [ ]:
# # Dense NNLSR with scipy.optimize.least_squares
# # here we prevented photons from being chosen using an internal pad
# this is slow
# 
# print(trace.shape, discretized_kernel.shape)
# c = np.concatenate((discretized_kernel, np.zeros(trace.size - discretized_kernel.size)), axis=0)
# C = circulant(c)
# print(C.shape)
# 
# def residual(x, C, trace):
#     assert type(x) is np.ndarray
#     assert type(C) is np.ndarray
#     assert type(trace) is np.ndarray
#     out = trace - C @ x
#     return out
# 
# # TODO these are not quite correct... needs to be loss = (residual)**2 + L1(weights) not L1(residual)
# # looking in trf.py, it seems we actually cant do L1 regularization, because the internal functions seperate out
# # residual and loss calcualtions, where we essential need y an X in the same function together. Could copy here and
# # modify... (commit on new branch before modifying) or could try to use scipt.omtimize.minimize like here:
# # https://stats.stackexchange.com/questions/573631/linear-regression-with-lasso-regularization-by-using-scikitlearn-and-scipy-optim
# # this might be simpler anyway, with more options
# # note in the above solution, they got around the problem by supplying y as a global, alongside the normal function arg X (Z, there).
# 
# def not_l1(z):    
#     loss = z**2 + z
#     d_loss = 2 * z + np.ones_like(z)
#     dd_loss = 2 * np.ones_like(z)
#     return np.stack((loss, d_loss, dd_loss)) 
# 
# solution1 = least_squares(fun=residual,
#                          jac='cs',
#                          method='trf', # use with bounds!
#                          bounds=[0, np.inf],
#                          args=(C, trace),
#                          x0=x0,
#                          x_scale=10, # it seems that a larger diff step + x_scale='jac' together can be an issue
#                          diff_step=.1,
#                          jac_sparsity=C,   # not necessary, but speeds things up a lot which will be useful for scaling to high n
#                          ftol=1E-10,
#                          loss=l1, # if not callable, 'linear', 'soft_l1' or 'huber'
#                          max_nfev=200,
#                          verbose=2
#                          )